In [4]:
import requests
import json
import pprint
import pandas as pd
from datetime import datetime
import pprint
import time
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; WOW64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/74.0.3729.169 YaBrowser/19.6.1.153 Yowser/2.5 Safari/537.36'}
df_tyler = pd.read_csv("tyler_holders_filtered.csv")
df_diddy = pd.read_csv("diddy_holders_filtered.csv")

In [9]:
tyler_top_10 = df_tyler["wallet"].head(10).tolist()
diddy_top_10 = df_diddy["wallet"].head(10).tolist()

all_top_wallets = list(set(tyler_top_10 + diddy_top_10))

master_list = []

def get_trade_history(wallet_address, limit=1000):
    #first we make sure the address is correctly formatted
    wallet_address = wallet_address.strip()
    #and that it exists
    if not wallet_address or wallet_address == "0x...":
        print("!!NO WALLET FOUND!!")
        return

    url = "https://data-api.polymarket.com/activity"
    params = {
        'user': wallet_address,
        'limit': limit,
    }

    print(f"requesting data for {wallet_address[:10]}")

    try:
        response = requests.get(url, params=params, headers=headers)
        print(response.status_code)
        data = response.json()

        if not data:
            print("NOTHING FOUND. RETURN TO BASE")
            return None

        #make a empty list to host the info
        clean_list = []
        for entry in data:
            raw_ts = entry.get('timestamp')
            #Format it correctly
            if isinstance(raw_ts, str):
                dt_obj = datetime.fromisoformat(raw_ts.replace('Z', '+00:00'))
            elif isinstance(raw_ts, (int, float)):
                if raw_ts > 10**11: raw_ts /= 1000
                dt_obj = datetime.fromtimestamp(raw_ts)
            else:
                continue

            dt_str = dt_obj.strftime('%Y-%m-%d %H:%M')
            
            #Get the numbers
            shares = float(entry.get('size', 0.0))
            price = float(entry.get('price', 0.0))
            
            #build our columns
            clean_list.append({
                'date': dt_str,
                'trade_type': entry.get('type', 'TRADE'),
                'side': entry.get('side', ''),
                'outcome': entry.get('outcome', ''),
                'amount': shares,
                'price': price,
                'total': price * shares,
                'title': entry.get('title', ''),
                'user': entry.get('proxyWallet', ''),
            })

        return clean_list

    except Exception as e:
        print(f"something went wrong: {e}.")
        return None

for i, wallet in enumerate(all_top_wallets):
    user_trades = get_trade_history(wallet, limit=1000)

    if user_trades:
        master_list.extend(user_trades)
    time.sleep(0.5)

if master_list:
    df_all_trades = pd.DataFrame(master_list)
    df_all_trades.to_csv("top_20_trade_history.csv")
    print("Success. Here is the dataframe:")
    print(df_all_trades.tail())

requesting data for 0x96f795d0
200
requesting data for 0x1f9e15f3
200
requesting data for 0x2e7f06e8
200
requesting data for 0xa7a6cd53
200
requesting data for 0x00d1bfdb
200
requesting data for 0xcf7c3d0c
200
requesting data for 0xccdac3c7
200
requesting data for 0x477a7d28
200
requesting data for 0xc1b02266
200
requesting data for 0xe4de861d
200
requesting data for 0x44fccfc4
200
requesting data for 0x157fa171
200
requesting data for 0xf6d8e74e
200
requesting data for 0x65bf3bfd
200
requesting data for 0x2b9dbf4b
200
requesting data for 0x7fda167d
200
requesting data for 0xd8163931
200
requesting data for 0xf44dcf50
200
requesting data for 0xc8ab97a9
200
requesting data for 0x00fa1da7
200
Success. Here is the dataframe:
                   date trade_type side outcome  amount  price     total  \
13524  2024-10-06 06:11      TRADE  BUY     Yes  500.00   0.30  150.0000   
13525  2024-10-06 06:11      TRADE  BUY     Yes   30.00   0.30    9.0000   
13526  2024-10-06 06:11      TRADE  BUY 

In [14]:
df_all_trades.shape

(13529, 9)

In [15]:
df_all_trades.columns

Index(['date', 'trade_type', 'side', 'outcome', 'amount', 'price', 'total',
       'title', 'user'],
      dtype='str')

In [17]:
df_all_trades

,date,trade_type,side,outcome,amount,price,total,title,user
0,2026-07-10 23:13,TRADE,BUY,No,2103.94000,0.53,1115.088200,Tyler Robinson convicted of homicide?,0x96f795d0821b75a1c7f886b2c1cd49d108b7d6ae
1,2026-07-10 20:45,MAKER_REBATE,,,55.61030,0.00,0.000000,,0x96f795d0821b75a1c7f886b2c1cd49d108b7d6ae
2,2026-07-10 20:13,YIELD,,,0.19050,0.00,0.000000,,0x96f795d0821b75a1c7f886b2c1cd49d108b7d6ae
3,2026-07-10 20:11,TRADE,BUY,No,44.68085,0.53,23.680851,Tyler Robinson convicted of homicide?,0x96f795d0821b75a1c7f886b2c1cd49d108b7d6ae
4,2026-07-10 20:10,TAKER_REBATE,,,1.13680,0.00,0.000000,,0x96f795d0821b75a1c7f886b2c1cd49d108b7d6ae
...,...,...,...,...,...,...,...,...,...
13524,2024-10-06 06:11,TRADE,BUY,Yes,500.00000,0.30,150.000000,Will Guilherme Boulos win the 2024 São Paulo m...,0x00fa1da7d2b8baa703a1d3554b5d1338be9030be
13525,2024-10-06 06:11,TRADE,BUY,Yes,30.00000,0.30,9.000000,Will Guilherme Boulos win the 2024 São Paulo m...,0x00fa1da7d2b8baa703a1d3554b5d1338be9030be
13526,2024-10-06 06:11,TRADE,BUY,Yes,500.00000,0.35,175.000000,Will Pablo Marçal win the 2024 São Paulo mayor...,0x00fa1da7d2b8baa703a1d3554b5d1338be9030be
13527,2024-10-06 06:11,TRADE,BUY,Yes,965.49000,0.35,337.921500,Will Pablo Marçal win the 2024 São Paulo mayor...,0x00fa1da7d2b8baa703a1d3554b5d1338be9030be


In [66]:
#Let's see how much these users spent on bets on average
df_bet_money = df_all_trades[df_all_trades["side"] == "BUY"]
df_bets_sum = df_bet_money.groupby("user")["total"].sum()
#let's get this to a csv so we can plot it (without the average)
print(f"On average, these top betters spent {round(df_bets_sum.mean())} dollars on their last 1000 bets.")

On average, these top betters spent 62121 dollars on their last 1000 bets.


In [49]:
df_bets_chart = df_bet_money[["outcome", "total", "title"]]
#This one is not ready yet...
df_bets_chart.to_csv("scatterplot_avrg_bet_per_user.csv", index=False)

In [47]:
df_bets_sum = pd.DataFrame(df_bets_sum)
df_bets_sum.to_csv("sum_of_top20_bet_history.csv", index=False)

In [52]:
#Okay let's make a dataframe which we can analyse using AI in order to categorize which categories they are betting on
#we take the one with only purchases
#Actually let's check first which is the most common individual bet
df_bet_money.value_counts("title").head(10)

title
Will Charles Leclerc be the 2025 Drivers Champion?                                                                    372
Will Google have the best AI model at the end of July 2026?                                                           253
Will Uhlsport be the brand of gloves worn by the Golden Glove winner at the 2026 FIFA World Cup?                      237
US forces enter Iran by December 31?                                                                                  178
Will Trump nominate Judy Shelton as the next Fed chair?                                                               156
Argentina vs. Egypt: Argentina O/U 5.5 Corners                                                                        127
Will the Liberal Democratic Party of Russia (LDPR) gain the most seats in the next Russian parliamentary election?    122
Brazil vs. Norway: Brazil O/U 5.5 Corners                                                                             110
Will Zelenskyy wea

In [57]:
#Now we check if the top ones is just one or two traders or if there is market consensus making these trades popular
cons_bets = df_bet_money.groupby("title")["user"].nunique()
cons_bets.sort_values(ascending=False).head(12)

title
Tyler Robinson convicted of homicide?                                           9
Diddy found guilty of sex trafficking?                                          6
Will Troy Jackson be the Maine Senate Democratic nominee on July 27?            4
Will Claude Fable 5 be restored for US customers by June 30?                    4
US x Iran permanent peace deal by May 31, 2026?                                 4
US x Iran permanent peace deal by June 15, 2026?                                4
Starmer out by June 30, 2026?                                                   4
Will the chopsticks catch SpaceX Starship Flight Test 12 Superheavy booster?    3
US forces enter Iran by March 31?                                               3
Will Claude Fable 5 be restored for US customers by July 1?                     3
Will Ronaldo Cry at the World Cup?                                              3
MicroStrategy sells any Bitcoin by May 31, 2026?                                3
Name: user

In [58]:
#Since we put a limit on the API for 1000 trades per user, it seems many
#of the whales has traded more than 1000 times since they bought tyler or diddy
#hence why they aren't at 10 each
print("The most popular market for these traders (besides the ones analyzed) is the US - Iran war, with three different bets being on the top 10, including a bet from earlier this year on if U.S forces would enter Iran. Also popular is wether Troy Jackson will be the Maine Senate Democratic nominee on July 27?, with four of the 20 betting on it. 'Will Ronaldo cry at the world Cup?' came in on number 9.")

The most popular market for these traders (besides the ones analyzed) is the US - Iran war, with three different bets being on the top 10, including a bet from earlier this year on if U.S forces would enter Iran. Also popular is wether Troy Jackson will be the Maine Senate Democratic nominee on July 27?, with four of the 20 betting on it. 'Will Ronaldo cry at the world Cup?' came in on number 9.


In [60]:
#Let's redo this with only the Tyler Robinson betters and see what we get.
#We make a new API call
tyler_top_100 = df_tyler["wallet"].head(100).tolist()

master_list = []

def get_trade_history(wallet_address, limit=1000):
    #first we make sure the address is correctly formatted
    wallet_address = wallet_address.strip()
    #and that it exists
    if not wallet_address or wallet_address == "0x...":
        print("!!NO WALLET FOUND!!")
        return

    url = "https://data-api.polymarket.com/activity"
    params = {
        'user': wallet_address,
        'limit': limit,
    }

    print(f"requesting data for {wallet_address[:10]}")

    try:
        response = requests.get(url, params=params, headers=headers)
        print(response.status_code)
        data = response.json()

        if not data:
            print("NOTHING FOUND. RETURN TO BASE")
            return None

        #make a empty list to host the info
        clean_list = []
        for entry in data:
            raw_ts = entry.get('timestamp')
            #Format it correctly
            if isinstance(raw_ts, str):
                dt_obj = datetime.fromisoformat(raw_ts.replace('Z', '+00:00'))
            elif isinstance(raw_ts, (int, float)):
                if raw_ts > 10**11: raw_ts /= 1000
                dt_obj = datetime.fromtimestamp(raw_ts)
            else:
                continue

            dt_str = dt_obj.strftime('%Y-%m-%d %H:%M')
            
            #Get the numbers
            shares = float(entry.get('size', 0.0))
            price = float(entry.get('price', 0.0))
            
            #build our columns
            clean_list.append({
                'date': dt_str,
                'trade_type': entry.get('type', 'TRADE'),
                'side': entry.get('side', ''),
                'outcome': entry.get('outcome', ''),
                'amount': shares,
                'price': price,
                'total': price * shares,
                'title': entry.get('title', ''),
                'user': entry.get('proxyWallet', ''),
            })

        return clean_list

    except Exception as e:
        print(f"something went wrong: {e}.")
        return None

for i, wallet in enumerate(tyler_top_100):
    user_trades = get_trade_history(wallet, limit=1000)

    if user_trades:
        master_list.extend(user_trades)
    time.sleep(0.5)

if master_list:
    df_tyler_traders_history = pd.DataFrame(master_list)
    df_tyler_traders_history.to_csv("tyler_traders_history.csv")
    print("Success. Here is the dataframe:")
    print(df_tyler_traders_history.tail())

requesting data for 0xe4de861d
200
requesting data for 0xa7a6cd53
200
requesting data for 0xf44dcf50
200
requesting data for 0x1f9e15f3
200
requesting data for 0x44fccfc4
200
requesting data for 0x00d1bfdb
200
requesting data for 0xc8ab97a9
200
requesting data for 0x2b9dbf4b
200
requesting data for 0x96f795d0
200
requesting data for 0xd8163931
200
requesting data for 0xac4a1fab
200
requesting data for 0xfc1490c0
200
requesting data for 0x7e31a5e9
200
requesting data for 0x760907c8
200
requesting data for 0x0b644e44
200
requesting data for 0x524db836
200
requesting data for 0xcf6a7146
200
requesting data for 0x7e35a2a8
200
requesting data for 0x5fa44ddb
200
requesting data for 0x9ad091ca
200
requesting data for 0xe6fd1806
200
requesting data for 0xbfbb9bdd
200
requesting data for 0xf17cd97d
200
requesting data for 0xd5b97d08
200
requesting data for 0x38390d45
200
requesting data for 0x3052776d
200
requesting data for 0xae2b0aae
200
requesting data for 0x15fbd312
200
requesting data for 

In [113]:
df_tyler_traders_history.shape

(73324, 9)

In [118]:
df_bet_money_tyl = df_tyler_traders_history[df_tyler_traders_history["side"] == "BUY"]
df_bets_sum_tyl = df_bet_money_tyl.groupby("user")["total"].sum()
df_bet_money_tyl.to_csv("tyler_trade_buys_only.csv", index=False)
print(df_bet_money_tyl.shape)
#let's get this to a csv so we can plot it (without the average)
df_bets_sum_tyl.mean()

(41291, 9)


np.float64(38681.72531863396)

In [156]:
print("These are the most popular bets for top 100 Tyler holders:")
df_bet_money_tyl.value_counts("title").head(10).to_csv("10_most_pop_bets.csv")

These are the most popular bets for top 100 Tyler holders:


In [163]:
df_bet_money_tyl.value_counts("title").head(10)

title
Will Alex Bores win the NY-12 Democratic Primary by less than 5%?                                                     540
Will the U.S. invade Iran before 2027?                                                                                489
Will Ronaldo Cry at the World Cup?                                                                                    456
Tyler Robinson convicted of homicide?                                                                                 364
Will Google have the best AI model at the end of July 2026?                                                           307
Will the Liberal Democratic Party of Russia (LDPR) gain the most seats in the next Russian parliamentary election?    304
Will Uhlsport be the brand of gloves worn by the Golden Glove winner at the 2026 FIFA World Cup?                      236
Will OpenAI release a new frontier model on or before September 30, 2026?                                             197
Will Pope Leo XIV 

In [137]:
#How are these popular bets distributed by yes and no?
#Make new df with most popular bets AND yes/no
df_yes_no = df_bet_money_tyl[df_bet_money_tyl["outcome"].isin(["Yes", "No"])]
df_popmarket = pd.crosstab(df_yes_no["title"], df_yes_no["outcome"])
df_popmarket["total_bets"] = df_popmarket.sum(axis=1)

In [153]:
print(f"Total unique markets: {df_yes_no['title'].nunique()}")

Total unique markets: 5468


In [157]:
df_popmarket = df_popmarket.sort_values(by="total_bets", ascending=False)
df_popmarket.to_csv("10_most_pop_bets_yesno.csv")

In [160]:
cons_bets_tyl = df_bet_money_tyl.groupby("title")["user"].nunique()
print(cons_bets_tyl.sort_values(ascending=False).head(51))
print(20, " of ", 50, "trades were about the USA-Iran war.")
print("25 percent had bet on if the Strait of Hormuz traffic returns to normal by July 31?")
pop_50_bets = cons_bets_tyl.sort_values(ascending=False).head(51)
df_50_pop_bets = pd.DataFrame(most_50_pop_bets)
df_50_pop_bets["percent"] = df_50_pop_bets["user"]

title
Tyler Robinson convicted of homicide?                                     87
Will France win the 2026 FIFA World Cup?                                  28
Strait of Hormuz traffic returns to normal by July 31?                    25
Will Ronaldo Cry at the World Cup?                                        23
Will Argentina win the 2026 FIFA World Cup?                               21
US x Iran diplomatic meeting by July 31, 2026?                            20
Will Mitch McConnell step down from the Senate before his term ends?      20
Mojtaba Khamenei seen in public by July 15?                               18
Will Count Binface win the Clacton by-election?                           17
Will Iran announce withdrawal from MOU negotiations by July 31?           17
Will Spain win the 2026 FIFA World Cup?                                   15
 Iran agrees to end enrichment of uranium by June 30?                     15
Will Argentina win on 2026-07-07?                                     

In [159]:
df_50_pop_bets.to_csv("50_pop_bets.csv")

In [109]:
df_50_pop_bets = df_50_pop_bets.drop(columns="user")
df_50_pop_bets

,percent
title,
Tyler Robinson convicted of homicide?,87
Will France win the 2026 FIFA World Cup?,28
Strait of Hormuz traffic returns to normal by July 31?,25
Will Ronaldo Cry at the World Cup?,23
Will Argentina win the 2026 FIFA World Cup?,21
"US x Iran diplomatic meeting by July 31, 2026?",20
Will Mitch McConnell step down from the Senate before his term ends?,20
Mojtaba Khamenei seen in public by July 15?,18
Will Count Binface win the Clacton by-election?,17


In [115]:
df_50_pop_bets.to_csv("50_most_popular_bets_percentage.csv")

In [80]:
df_tyl_alien_buys = df_bet_money_tyl[df_bet_money_tyl["title"] == "Will the US confirm that aliens exist before 2027?"]

In [87]:
df_tyl_alien_yes = df_tyl_alien_buys[df_tyl_alien_buys["outcome"] == "Yes"]
print(f"There are {len(df_tyl_alien_buys)} purchases of the alien bet.")
print(f"{len(df_tyl_alien_yes)} of them were betting yes.")

There are 138 purchases of the alien bet.
15 of them were betting yes.


In [ ]:
#Now we build an AI categorization for the what the top holders bet on.